# Meta Ads — Silver Transform

Bronze → Silver for all Meta entity + insight extracts:
- Flatten `raw_json`
- Enforce types / natural keys
- Deduplicate
- Explode insight actions
- Write Delta tables under `silver.*`

**Incremental:** default `FULL_REFRESH=False` merges new dates (lookback 2 days). Existing history is kept.



In [ ]:
bronze_schema = "bronze"
silver_schema = "silver"
run_optimize = True

# Incremental (default): merge new dates; do not wipe history
FULL_REFRESH = False
INCREMENTAL_LOOKBACK_DAYS = 2
INCREMENTAL_MAX_DAYS = 14



In [ ]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable
from pyspark.sql.types import ArrayType, DoubleType, MapType, StringType, StructField, StructType

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

action_schema = ArrayType(
    StructType(
        [
            StructField("action_type", StringType(), True),
            StructField("value", StringType(), True),
        ]
    )
)


def read_bronze(table: str):
    return spark.table(f"{bronze_schema}.{table}")


def with_raw(df):
    return df.withColumn("_raw", F.from_json(F.col("raw_json"), MapType(StringType(), StringType())))


def cents_to_amount(col):
    return (F.col(col).cast("double") / F.lit(100.0)).alias(col.replace("_raw", "") + "_amount" if False else col)


def dedup(df, keys):
    w = Window.partitionBy(*keys).orderBy(F.col("ingestion_time").desc_nulls_last())
    return (
        df.withColumn("_rn", F.row_number().over(w))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )


# Incremental helpers (controls: FULL_REFRESH / LOOKBACK / MAX_DAYS in parameters cell)
from delta.tables import DeltaTable

def _table_exists(name: str) -> bool:
    try:
        spark.table(name).limit(1).collect()
        return True
    except Exception:
        return False

def _path_is_delta(path: str) -> bool:
    try:
        return DeltaTable.isDeltaTable(spark, path)
    except Exception:
        return False

def _max_date(table_or_path: str, date_col: str, is_path: bool = False):
    try:
        df = spark.read.format("delta").load(table_or_path) if is_path else spark.table(table_or_path)
        return df.agg(F.max(F.col(date_col)).alias("m")).collect()[0]["m"]
    except Exception:
        return None

def filter_by_watermark(df, date_col: str, target: str, is_path: bool = False):
    if FULL_REFRESH:
        print(f"[FULL_REFRESH] no watermark filter: {target}")
        return df
    exists = _path_is_delta(target) if is_path else _table_exists(target)
    if not exists:
        print(f"[INCR] target missing → first load: {target}")
        return df
    wm = _max_date(target, date_col, is_path=is_path)
    if wm is None:
        print(f"[INCR] empty watermark → full batch: {target}")
        return df
    out = df.filter(F.col(date_col).isNotNull() & (F.col(date_col) >= F.date_sub(F.lit(wm), int(INCREMENTAL_LOOKBACK_DAYS))))
    st = out.agg(F.min(date_col).alias("mn"), F.max(date_col).alias("mx"), F.count(F.lit(1)).alias("n")).collect()[0]
    print(f"[INCR] {target} wm={wm} lookback={INCREMENTAL_LOOKBACK_DAYS}d rows={st['n']} range={st['mn']}..{st['mx']}")
    if st["n"] and st["mn"] is not None and st["mx"] is not None:
        span = (st["mx"] - st["mn"]).days
        if span > int(INCREMENTAL_MAX_DAYS):
            raise ValueError(
                f"Incremental batch for {target} spans {span} days (> {INCREMENTAL_MAX_DAYS}). "
                "Refusing large backfill. Use daily/2-day bronze, or set FULL_REFRESH=True intentionally."
            )
    return out

def merge_or_overwrite_table(df, target: str, keys, partition_cols=None, stamp_col="silver_processed_at"):
    if stamp_col and stamp_col not in df.columns:
        df = df.withColumn(stamp_col, F.current_timestamp())
    df = df.dropDuplicates(list(keys))
    if FULL_REFRESH or not _table_exists(target):
        w = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        if partition_cols:
            w = w.partitionBy(*partition_cols)
        w.saveAsTable(target)
        mode = "OVERWRITE"
    else:
        cond = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
        (DeltaTable.forName(spark, target).alias("t").merge(df.alias("s"), cond)
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        mode = "MERGE"
    print(f"[OK] {target} ({mode}) total={spark.table(target).count():,}")

def merge_or_overwrite_path(df, path: str, keys, partition_cols=None, stamp_col="_processed_at"):
    if stamp_col and stamp_col not in df.columns:
        df = df.withColumn(stamp_col, F.current_timestamp())
    df = df.dropDuplicates(list(keys))
    if FULL_REFRESH or not _path_is_delta(path):
        w = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").option("mergeSchema", "true")
        if partition_cols:
            w = w.partitionBy(*partition_cols)
        w.save(path)
        mode = "OVERWRITE"
    else:
        cond = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
        (DeltaTable.forPath(spark, path).alias("t").merge(df.alias("s"), cond)
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        mode = "MERGE"
    print(f"[OK] {path} ({mode}) total={spark.read.format('delta').load(path).count():,}")

def write_silver(df, table: str, keys, partition_cols=None, date_col=None):
    """Incremental MERGE by keys. If date_col set, only watermark window is merged."""
    target = f"{silver_schema}.{table}"
    batch = df
    if date_col:
        batch = filter_by_watermark(batch, date_col, target)
        if len(batch.take(1)) == 0:
            print(f"[SKIP] {target}: no new incremental rows")
            return 0
    merge_or_overwrite_table(batch, target, keys, partition_cols=partition_cols, stamp_col="silver_processed_at")
    if run_optimize:
        spark.sql(f"OPTIMIZE {target}")
    return spark.table(target).count()



## Campaigns / Ad sets / Ads

In [ ]:
# Campaigns — parse raw_json with a typed schema for nested arrays
campaign_schema = StructType(
    [
        StructField("id", StringType(), True),
        StructField("account_id", StringType(), True),
        StructField("name", StringType(), True),
        StructField("objective", StringType(), True),
        StructField("status", StringType(), True),
        StructField("configured_status", StringType(), True),
        StructField("effective_status", StringType(), True),
        StructField("buying_type", StringType(), True),
        StructField("bid_strategy", StringType(), True),
        StructField("daily_budget", StringType(), True),
        StructField("lifetime_budget", StringType(), True),
        StructField("budget_remaining", StringType(), True),
        StructField("special_ad_categories", ArrayType(StringType()), True),
        StructField("created_time", StringType(), True),
        StructField("updated_time", StringType(), True),
        StructField("start_time", StringType(), True),
        StructField("stop_time", StringType(), True),
    ]
)

camp_b = read_bronze("meta_campaigns").withColumn("j", F.from_json("raw_json", campaign_schema))
campaigns = (
    camp_b.select(
        F.coalesce(F.col("j.id"), F.col("entity_id")).alias("campaign_id"),
        F.col("j.name").alias("campaign_name"),
        F.col("j.account_id").alias("meta_account_id"),
        F.col("j.objective").alias("objective"),
        F.col("j.status").alias("status"),
        F.col("j.configured_status").alias("configured_status"),
        F.col("j.effective_status").alias("effective_status"),
        F.col("j.buying_type").alias("buying_type"),
        F.col("j.bid_strategy").alias("bid_strategy"),
        (F.col("j.daily_budget").cast("double") / 100.0).alias("daily_budget_amount"),
        (F.col("j.lifetime_budget").cast("double") / 100.0).alias("lifetime_budget_amount"),
        (F.col("j.budget_remaining").cast("double") / 100.0).alias("budget_remaining_amount"),
        F.to_json(F.col("j.special_ad_categories")).alias("special_ad_categories"),
        F.col("j.created_time").alias("created_time"),
        F.col("j.updated_time").alias("updated_time"),
        F.col("j.start_time").alias("start_time"),
        F.col("j.stop_time").alias("stop_time"),
        "connector_id",
        "tenant_id",
        "account_id",
        "account_name",
        "platform",
        F.col("batch_id").alias("source_batch_id"),
        F.to_timestamp("ingestion_time").alias("ingestion_time"),
        "extraction_start_date",
        "extraction_end_date",
    )
    .filter(F.col("campaign_id").isNotNull())
)
campaigns = dedup(campaigns, ["campaign_id"])
write_silver(campaigns, "meta_campaigns", keys=["campaign_id"])


In [ ]:
adset_schema = StructType(
    [
        StructField("id", StringType(), True),
        StructField("campaign_id", StringType(), True),
        StructField("name", StringType(), True),
        StructField("status", StringType(), True),
        StructField("optimization_goal", StringType(), True),
        StructField("billing_event", StringType(), True),
        StructField("bid_strategy", StringType(), True),
        StructField("daily_budget", StringType(), True),
        StructField("lifetime_budget", StringType(), True),
        StructField("created_time", StringType(), True),
        StructField("updated_time", StringType(), True),
        StructField("targeting", StringType(), True),  # keep as JSON string then re-parse map
    ]
)

# targeting is nested object — parse full raw_json as string map then extract via get_json_object
adsets_b = read_bronze("meta_adsets")
adsets = (
    adsets_b.select(
        F.coalesce(F.get_json_object("raw_json", "$.id"), F.col("entity_id")).alias("adset_id"),
        F.coalesce(F.get_json_object("raw_json", "$.campaign_id"), F.col("parent_entity_id")).alias("campaign_id"),
        F.get_json_object("raw_json", "$.name").alias("adset_name"),
        F.get_json_object("raw_json", "$.status").alias("status"),
        F.get_json_object("raw_json", "$.optimization_goal").alias("optimization_goal"),
        F.get_json_object("raw_json", "$.billing_event").alias("billing_event"),
        F.get_json_object("raw_json", "$.bid_strategy").alias("bid_strategy"),
        (F.get_json_object("raw_json", "$.daily_budget").cast("double") / 100.0).alias("daily_budget_amount"),
        (F.get_json_object("raw_json", "$.lifetime_budget").cast("double") / 100.0).alias("lifetime_budget_amount"),
        F.get_json_object("raw_json", "$.targeting.age_min").cast("int").alias("age_min"),
        F.get_json_object("raw_json", "$.targeting.age_max").cast("int").alias("age_max"),
        F.get_json_object("raw_json", "$.targeting.genders").alias("genders"),
        F.get_json_object("raw_json", "$.targeting.geo_locations.countries").alias("geo_countries"),
        F.get_json_object("raw_json", "$.targeting").alias("targeting_json"),
        F.get_json_object("raw_json", "$.created_time").alias("created_time"),
        F.get_json_object("raw_json", "$.updated_time").alias("updated_time"),
        "connector_id",
        "tenant_id",
        "account_id",
        "account_name",
        "platform",
        F.col("batch_id").alias("source_batch_id"),
        F.to_timestamp("ingestion_time").alias("ingestion_time"),
        "extraction_start_date",
        "extraction_end_date",
    )
    .filter(F.col("adset_id").isNotNull() & F.col("campaign_id").isNotNull())
)
adsets = dedup(adsets, ["adset_id"])
write_silver(adsets, "meta_adsets", keys=["adset_id"])


In [ ]:
ads_b = read_bronze("meta_ads")
ads = (
    ads_b.select(
        F.coalesce(F.get_json_object("raw_json", "$.id"), F.col("entity_id")).alias("ad_id"),
        F.coalesce(F.get_json_object("raw_json", "$.adset_id"), F.col("parent_entity_id")).alias("adset_id"),
        F.get_json_object("raw_json", "$.campaign_id").alias("campaign_id"),
        F.get_json_object("raw_json", "$.name").alias("ad_name"),
        F.get_json_object("raw_json", "$.status").alias("status"),
        F.get_json_object("raw_json", "$.effective_status").alias("effective_status"),
        F.get_json_object("raw_json", "$.creative.id").alias("creative_id"),
        F.get_json_object("raw_json", "$.creative.thumbnail_url").alias("creative_thumbnail_url"),
        F.get_json_object("raw_json", "$.preview_shareable_link").alias("preview_shareable_link"),
        F.get_json_object("raw_json", "$.tracking_specs").alias("tracking_specs_json"),
        F.get_json_object("raw_json", "$.created_time").alias("created_time"),
        F.get_json_object("raw_json", "$.updated_time").alias("updated_time"),
        "connector_id",
        "tenant_id",
        "account_id",
        "account_name",
        "platform",
        F.col("batch_id").alias("source_batch_id"),
        F.to_timestamp("ingestion_time").alias("ingestion_time"),
        "extraction_start_date",
        "extraction_end_date",
    )
    .filter(F.col("ad_id").isNotNull() & F.col("adset_id").isNotNull())
)
ads = dedup(ads, ["ad_id"])
write_silver(ads, "meta_ads", keys=["ad_id"])


## Daily insights + exploded actions

In [ ]:
metric_cols = [
    "impressions",
    "reach",
    "frequency",
    "clicks",
    "unique_clicks",
    "inline_link_clicks",
    "spend",
    "cpc",
    "cpm",
    "cpp",
    "ctr",
    "unique_ctr",
]


def insight_metric_exprs():
    return [(F.get_json_object("raw_json", f"$.{c}").cast("double").alias(c)) for c in metric_cols]


adset_ins_b = read_bronze("meta_adset_insights")
adset_insights = (
    adset_ins_b.select(
        F.coalesce(F.get_json_object("raw_json", "$.adset_id"), F.col("entity_id")).alias("adset_id"),
        F.get_json_object("raw_json", "$.campaign_id").alias("campaign_id"),
        F.get_json_object("raw_json", "$.date_start").alias("date_start"),
        F.get_json_object("raw_json", "$.date_stop").alias("date_stop"),
        *insight_metric_exprs(),
        "connector_id",
        "tenant_id",
        "account_id",
        "account_name",
        "platform",
        F.col("batch_id").alias("source_batch_id"),
        F.to_timestamp("ingestion_time").alias("ingestion_time"),
        "extraction_start_date",
        "extraction_end_date",
    )
    .filter(F.col("adset_id").isNotNull() & F.col("date_start").isNotNull())
)
adset_insights = dedup(adset_insights, ["adset_id", "date_start"])
write_silver(adset_insights, "meta_adset_insights_daily", keys=["adset_id", "date_start"], partition_cols=["date_start"], date_col="date_start")

ad_ins_b = read_bronze("meta_ad_insights")
ad_insights = (
    ad_ins_b.select(
        F.coalesce(F.get_json_object("raw_json", "$.ad_id"), F.col("entity_id")).alias("ad_id"),
        F.get_json_object("raw_json", "$.ad_name").alias("ad_name"),
        F.get_json_object("raw_json", "$.adset_id").alias("adset_id"),
        F.get_json_object("raw_json", "$.adset_name").alias("adset_name"),
        F.get_json_object("raw_json", "$.campaign_id").alias("campaign_id"),
        F.get_json_object("raw_json", "$.campaign_name").alias("campaign_name"),
        F.get_json_object("raw_json", "$.date_start").alias("date_start"),
        F.get_json_object("raw_json", "$.date_stop").alias("date_stop"),
        *insight_metric_exprs(),
        "connector_id",
        "tenant_id",
        "account_id",
        "account_name",
        "platform",
        F.col("batch_id").alias("source_batch_id"),
        F.to_timestamp("ingestion_time").alias("ingestion_time"),
        "extraction_start_date",
        "extraction_end_date",
    )
    .filter(F.col("ad_id").isNotNull() & F.col("date_start").isNotNull())
)
ad_insights = dedup(ad_insights, ["ad_id", "date_start"])
write_silver(ad_insights, "meta_ad_insights_daily", keys=["ad_id", "date_start"], partition_cols=["date_start"], date_col="date_start")


In [ ]:
def explode_actions(df, entity_type: str, id_json_path: str, id_fallback_col: str):
    parsed = df.withColumn("actions", F.from_json(F.get_json_object("raw_json", "$.actions"), action_schema))
    return (
        parsed.withColumn("action", F.explode_outer("actions"))
        .select(
            F.lit(entity_type).alias("entity_type"),
            F.coalesce(F.get_json_object("raw_json", id_json_path), F.col(id_fallback_col)).alias("entity_id"),
            F.get_json_object("raw_json", "$.campaign_id").alias("campaign_id"),
            F.get_json_object("raw_json", "$.adset_id").alias("adset_id"),
            F.get_json_object("raw_json", "$.ad_id").alias("ad_id"),
            F.get_json_object("raw_json", "$.date_start").alias("date_start"),
            F.get_json_object("raw_json", "$.date_stop").alias("date_stop"),
            F.col("action.action_type").alias("action_type"),
            F.col("action.value").cast("double").alias("action_value"),
            "connector_id",
            "tenant_id",
            "account_id",
            "account_name",
            "platform",
            F.col("batch_id").alias("source_batch_id"),
            F.to_timestamp("ingestion_time").alias("ingestion_time"),
        )
        .filter(F.col("entity_id").isNotNull() & F.col("date_start").isNotNull() & F.col("action_type").isNotNull())
    )


actions = explode_actions(adset_ins_b, "adset", "$.adset_id", "entity_id").unionByName(
    explode_actions(ad_ins_b, "ad", "$.ad_id", "entity_id")
)
actions = dedup(actions, ["entity_type", "entity_id", "date_start", "action_type"])
write_silver(actions, "meta_insight_actions", keys=["entity_type", "entity_id", "date_start", "action_type"], partition_cols=["date_start"], date_col="date_start")


In [ ]:
# Path verify (ops smoke table)
try:
    pv = read_bronze("meta_path_verify").withColumn("silver_processed_at", F.current_timestamp())
    write_silver(pv, "meta_path_verify", keys=["path"])
except Exception as exc:
    print(f"path_verify skipped: {exc}")

# Validation gate
print("\n=== Silver validation ===")
for t in [
    "meta_campaigns",
    "meta_adsets",
    "meta_ads",
    "meta_adset_insights_daily",
    "meta_ad_insights_daily",
    "meta_insight_actions",
]:
    print(f"{silver_schema}.{t}: {spark.table(f'{silver_schema}.{t}').count():,}")

orphan_adsets = spark.sql(
    f"""
    SELECT COUNT(*) AS c
    FROM {silver_schema}.meta_adsets a
    LEFT ANTI JOIN {silver_schema}.meta_campaigns c
      ON a.campaign_id = c.campaign_id
    """
).collect()[0][0]
orphan_ads = spark.sql(
    f"""
    SELECT COUNT(*) AS c
    FROM {silver_schema}.meta_ads a
    LEFT ANTI JOIN {silver_schema}.meta_adsets s
      ON a.adset_id = s.adset_id
    """
).collect()[0][0]
print(f"orphan adsets: {orphan_adsets}")
print(f"orphan ads: {orphan_ads}")
assert orphan_adsets == 0 and orphan_ads == 0, "Referential integrity failed"
